In [ ]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_root)
import torch 

from auto_circuit.data import load_datasets_from_json
from auto_circuit.experiment_utils import load_tl_model
from auto_circuit.prune_algos.mask_gradient import mask_gradient_prune_scores
from auto_circuit.types import PruneScores
from auto_circuit.utils.graph_utils import patchable_model
from auto_circuit.utils.misc import repo_path_to_abs_path
from auto_circuit.visualize import draw_seq_graph

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from transformers import PreTrainedTokenizerFast, AutoTokenizer
import transformer_lens as tl
from transformer_lens import HookedTransformer, HookedTransformerConfig
import json

%load_ext autoreload
%autoreload 2

In [ ]:
TOKENIZER_DIR  = "../model/wordlevel_tokenizer"

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR, add_bos_token=True)

# --- Load config ---
with open("../model/trained_transformerlens_model/config.json", "r") as f:
    cfg_dict = json.load(f)

# --- Fix dtype string back to actual torch dtype ---
if isinstance(cfg_dict.get("dtype"), str):
    cfg_dict["dtype"] = getattr(torch, cfg_dict["dtype"].replace("torch.", ""))

# --- Rebuild config and model ---
config = HookedTransformerConfig.from_dict(cfg_dict)
model = HookedTransformer(config)
model.load_state_dict(torch.load("../model/trained_transformerlens_model/model_weights.pth"))
model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

model.set_tokenizer(tokenizer)

model.cfg.default_prepend_bos, model.cfg.tokenizer_prepends_bos

model.set_use_hook_mlp_in(True)

model.set_use_attn_result(True)
model.set_use_attn_in(True)
model.set_use_split_qkv_input(True)
model.set_use_hook_mlp_in(True)

model.eval()

for param in model.parameters():
    param.requires_grad = False

In [ ]:
path = repo_path_to_abs_path("/home/iustin/Round_and_Round_RoPE/data/succession_augmented_last.json")

train_loader, test_loader = load_datasets_from_json(
    model=model,
    path=path,
    device=device,
    prepend_bos=False,
    tail_divergence=False,
    batch_size=1,
    train_test_size=(13, 13),
)

auto_model = patchable_model(
    model,
    factorized=True,
    slice_output="last_seq",
    separate_qkv=True,
    device=device,
)

attribution_scores: PruneScores = mask_gradient_prune_scores(
    model=auto_model,
    dataloader=train_loader,
    official_edges=None,
    grad_function="logit",
    answer_function="avg_diff",
    mask_val=0.0,
)

fig = draw_seq_graph(
    auto_model, attribution_scores, score_threshold=3.5, layer_spacing=True, orientation="v"
)
fig.write_image(repo_path_to_abs_path("/home/iustin/Round_and_Round_RoPE/auto_circuit_exps/test_last.png"), scale=4)

In [ ]:
tl.utils.test_prompt(prompt="the last term in the sequence 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 is", answer= '20', model=model, print_details=True)

In [ ]:
path = repo_path_to_abs_path("/home/iustin/Round_and_Round_RoPE/data/succession_augmented_next.json")

train_loader, test_loader = load_datasets_from_json(
    model=model,
    path=path,
    device=device,
    prepend_bos=False,
    tail_divergence=False,
    batch_size=1,
    train_test_size=(13, 13),
)

# model = patchable_model(
#     model,
#     factorized=True,
#     slice_output="last_seq",
#     separate_qkv=True,
#     device=device,
# )

attribution_scores: PruneScores = mask_gradient_prune_scores(
    model=auto_model,
    dataloader=train_loader,
    official_edges=None,
    grad_function="logit",
    answer_function="avg_diff",
    mask_val=0.0,
)

fig = draw_seq_graph(
    auto_model, attribution_scores, score_threshold=3.5, layer_spacing=True, orientation="v"
)
fig.write_image(repo_path_to_abs_path("/home/iustin/Round_and_Round_RoPE/auto_circuit_exps/test_next.png"), scale=4)

In [ ]:
tl.utils.test_prompt(prompt="the next term in the sequence 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 is", answer= '20', model=model, print_details=True)